# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides an example for loading and exploring a dataset using the `mlcroissant` library, referencing Croissant schema entities by their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

Dataset Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata object
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Identifier (@id): {metadata.id}")
print(f"Published on: {metadata.datePublished}")
print(f"Version: {metadata.version}")

## 2. Data Overview
Review available record sets, their fields, and corresponding entity (`@id`) values.

_Note: All references use the entity `@id` as per Croissant schema._

**Record Sets and Fields:**

In [ ]:
record_sets = dataset.record_sets
print(f"There are {len(record_sets)} record sets in this dataset.")

for rs in record_sets:
    print(f"\nRecordSet name: {rs.name}")
    print(f"RecordSet @id: {rs.id}")
    print(f"RecordSet description: {rs.description}")
    print("Fields:")
    for field in rs.fields:
        print(f"  - {field.name} (@id: {field.id}, type: {field.data_type})")

### Example record review
Let's print a few example records from each record set using the corresponding `@id`.

In [ ]:
for rs in record_sets:
    print(f"\n--- Records from RecordSet: {rs.name} (@id: {rs.id}) ---")
    for i, record in enumerate(dataset.records(record_set=rs.id)):
        print(record)
        if i >= 2:  # Show up to 3 examples
            break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

**All entities are referenced by their `@id`.**

In [ ]:
# List the RecordSet @ids
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"\nColumns in DataFrame for RecordSet @id={rs_id}:")
    print(df.columns.tolist())
    print(f"Sample records in DataFrame for RecordSet @id={rs_id}:")
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalization, grouping, and outlier removal.

We'll use entities referenced by their `@id`. Choose a numeric field and a group field from the RecordSet.

In [ ]:
# For example, select the first record set and its numeric fields
first_rs = record_sets[0]
first_rs_id = first_rs.id
df = dataframes[first_rs_id]

# Find a numeric field
numeric_fields = [f for f in first_rs.fields if f.data_type in ['Integer', 'Float', 'Number']]
if not numeric_fields:
    raise ValueError("No numeric fields in the first RecordSet.")

numeric_field = numeric_fields[0]
numeric_field_id = numeric_field.id
# Find a grouping field (categorical)
group_fields = [f for f in first_rs.fields if f.data_type in ['Text', 'Boolean']]

group_field_id = group_fields[0].id if group_fields else numeric_field_id

# Filter records with numeric field above a threshold
threshold = 10
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    mean_val = filtered_df[numeric_field_id].mean()
    std_val = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean_val) / std_val
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].tolist()].head())

    # Group by a chosen categorical field and show means
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"Numeric field {numeric_field_id} is not available in the DataFrame.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the numeric field (if present)
if numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Scatter plot with group field (if available and not numeric)
    if group_field_id in df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
- Successfully loaded and explored the FAIR\textsuperscript{2} dataset using the `mlcroissant` library.
- Reviewed metadata and record sets via their `@id`, loaded records into dataframes, and performed basic EDA and visualizations.
- All entities and columns referenced by `@id`—ensuring reproducibility and transparency.
- You can now extend this analysis with more advanced feature engineering or modeling as needed!